# Kramer Classifier — Demo Walkthrough

**What this is:** A prototype of the transfer-learning pipeline used for neonatal jaundice severity classification. Trained on the HAM10000 public dermoscopy dataset as a stand-in — same architecture and training techniques as the production classifier, different data.

**What this demonstrates:**
1. End-to-end image classification with transfer learning (ResNet50)
2. Handling severe class imbalance with class-weighted loss
3. Honest evaluation: held-out test set, calibration, fairness audit
4. Lighting robustness: diagnosing and partially fixing degradation under suboptimal lighting

**Runtime:** ~60 seconds (loads saved checkpoints, no training).

In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import torch
import torch.nn as nn
from sklearn.metrics import classification_report

from data import get_dataloaders
from model import build_model

DATA_DIR    = '../data'
RESULTS_DIR = Path('../results')
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 1 — The task

The **Kramer scale** describes cephalocaudal progression of jaundice in newborns across five zones (face → trunk → thighs → lower legs → palms/soles). Higher zone = higher bilirubin = greater clinical urgency. Visual Kramer assessment is the standard screening tool in resource-limited settings where transcutaneous bilirubinometers are unavailable.

We train on **HAM10000** — 10k adult dermoscopy images across 7 skin lesion classes — as a public proxy. The class distribution is severely imbalanced: `nv` (melanocytic nevi) is ~72% of training data; `df` (dermatofibroma) is under 2%. That imbalance mirrors the dynamic the production classifier faces, where most neonatal images are healthy/low-zone and high-severity zones are rare.

In [ ]:
_, _, test_loader = get_dataloaders(DATA_DIR, batch_size=32)
class_names = test_loader.dataset.classes
num_classes  = len(class_names)

train_dir = Path(DATA_DIR) / 'train'
counts = {cls: len(list((train_dir / cls).glob('*.jpg'))) for cls in class_names}

fig, ax = plt.subplots(figsize=(8, 3))
bars = ax.bar(counts.keys(), counts.values(), color='steelblue', alpha=0.85)
ax.bar_label(bars)
ax.set_ylabel('Training images')
ax.set_title('Class distribution — HAM10000 training split (5229 total)')
ax.axhline(5229 / num_classes, linestyle='--', color='gray', label='Balanced baseline')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Imbalance ratio: nv/df = {counts["nv"] / counts["df"]:.0f}×')

## 2 — Run 1: The imbalance problem

ResNet50, backbone frozen, no correction for class imbalance. The model discovers that predicting `nv` for everything yields ~80% val accuracy — higher than trying to learn the actual minority classes. That's not learning; it's exploiting the distribution.

| Metric | Run 1 |
|---|---|
| Val accuracy | 0.7964 |
| Macro-avg F1 | 0.44 |
| `df` recall | 0.18 |
| `vasc` recall | 0.07 |

`vasc` recall of 0.07 means the model catches 7% of true vasculoma cases. That's the failure mode.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].imshow(mpimg.imread(RESULTS_DIR / 'v1_training_curves.png'))
axes[0].axis('off')
axes[0].set_title('Run 1 — training curves', fontsize=11)
axes[1].imshow(mpimg.imread(RESULTS_DIR / 'v1_confusion_matrix.png'))
axes[1].axis('off')
axes[1].set_title('Run 1 — confusion matrix (val set)', fontsize=11)
plt.tight_layout()
plt.show()

## 3 — Run 2: Class-weighted loss

The fix: weight each class inversely by its frequency. A mistake on `df` (rare) costs 74× more than a mistake on `nv` (common). Nothing else changes — same architecture, same data, same learning rate.

Raw accuracy drops because the model stops cheating. Per-class recall on the minority classes jumps 2–9×.

| Metric | Run 1 | Run 2 | Change |
|---|---|---|---|
| Val accuracy | 0.7964 | 0.7027 | −9.4pp |
| Macro-avg F1 | 0.44 | 0.47 | +3pp |
| `df` recall | 0.18 | 0.55 | +37pp |
| `vasc` recall | 0.07 | 0.67 | +60pp |
| `mel` recall | 0.24 | 0.40 | +16pp |

The accuracy drop is the intended result. The model can't cheat anymore, so it has to earn accuracy by getting minority classes right.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].imshow(mpimg.imread(RESULTS_DIR / 'comparison.png'))
axes[0].axis('off')
axes[0].set_title('Run 1 vs Run 2 — val accuracy & loss', fontsize=11)
axes[1].imshow(mpimg.imread(RESULTS_DIR / 'v2_confusion_matrix.png'))
axes[1].axis('off')
axes[1].set_title('Run 2 — confusion matrix (val set)', fontsize=11)
plt.tight_layout()
plt.show()

## 4 — Held-out test evaluation

The test set has been untouched until this point. Running it once gives the honest number — no hyperparameter tuning on the test set, no peeking.

In [ ]:
model = build_model(num_classes=num_classes, backbone='resnet50', freeze_backbone=True)
model.load_state_dict(torch.load(RESULTS_DIR / 'v2' / 'best_model.pt', map_location=DEVICE))
model = model.to(DEVICE).eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        all_preds.extend(model(images).argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
test_acc   = (all_preds == all_labels).mean()

print(f'Test accuracy: {test_acc:.4f}  (val was 0.7027 — no overfitting)\n')
print(classification_report(all_labels, all_preds, target_names=class_names, digits=3))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(mpimg.imread(RESULTS_DIR / 'test_confusion_matrix.png'))
ax.axis('off')
plt.tight_layout()
plt.show()

## 5 — Calibration

Accuracy tells you how often the model is right. Calibration tells you whether the model's stated confidence is trustworthy — if it says 80%, is it actually right 80% of the time?

**ECE (Expected Calibration Error) = 0.069.** Lower is better; 0 is perfect. The model is **underconfident** — most points sit above the diagonal (accuracy exceeds confidence). This is the safer failure mode: an underconfident prediction triggers a human review, an overconfident wrong prediction doesn't.

The production gate is ECE ≤ 0.04. This prototype is roughly 2× over that bar. The standard fix is temperature scaling — dividing logits by a learned scalar before softmax. It doesn't change accuracy, only calibration.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(mpimg.imread(RESULTS_DIR / 'calibration_reliability.png'))
ax.axis('off')
plt.tight_layout()
plt.show()

## 6 — Lighting robustness

**The problem:** The classifier degrades under suboptimal lighting (underexposure, overexposure, color temperature shift). This is a training distribution mismatch — the model trained on well-controlled dermoscopy images and never saw lighting variation.

**Run 3** adds aggressive lighting augmentation to the training pipeline (brightness ±50%, contrast ±50%, hue shift, random autocontrast, Gaussian blur). Everything else — architecture, class weighting, learning rate — stays the same.

**The benchmark:** brightness-tertile accuracy. The 1121 test images are binned into dark/medium/light thirds by mean luminance. A smaller gap across bins = better robustness to real-world lighting conditions.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.imshow(mpimg.imread(RESULTS_DIR / 'lighting_robustness_comparison.png'))
ax.axis('off')
plt.tight_layout()
plt.show()

**Production roadmap for lighting robustness:**
1. Aggressive augmentation at training time — this notebook (done)
2. CLAHE illumination normalization at inference — no retraining needed
3. Test-time augmentation (TTA) — average predictions over lighting variants at inference
4. Real validation: neonatal images captured under controlled lighting variation

## 7 — Skin-tone fairness

HAM10000 ships no Fitzpatrick labels, so we use mean image luminance as a proxy and check whether performance differs across brightness tertiles. This is a technique demonstration, not a conclusion — in HAM10000, brightness mostly tracks lesion color rather than patient skin tone.

The 6.4pp accuracy gap between best and worst bins is the kind of number worth flagging for a real fairness audit. A real audit requires Fitzpatrick-labeled neonatal data.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(mpimg.imread(RESULTS_DIR / 'fairness_recall_by_tone.png'))
ax.axis('off')
plt.tight_layout()
plt.show()

## Summary

| Component | Result | Note |
|---|---|---|
| Architecture | ResNet50, frozen backbone | Transfer learning from ImageNet |
| Class imbalance | Class-weighted loss | `df` recall: 0.18 → 0.55 |
| Test accuracy | 0.71–0.72 | No overfitting (val ≈ test) |
| Macro-avg F1 | 0.453–0.474 | Penalizes minority-class misses |
| Calibration (ECE) | 0.069–0.083 | Underconfident — safer failure mode |
| Lighting robustness | Run 3 reduces brightness-bin gap | Augmentation fix demonstrated |
| Fairness | Technique pipeline in place | Proxy only; real audit needs Fitzpatrick labels |

**What this is not:** A deployable model. HAM10000 is adult dermoscopy, not neonatal jaundice. The architecture and pipeline are production-grade; the training data is a stand-in.

**What's next:**
- Temperature scaling to bring ECE ≤ 0.04
- Phase 3: 1D signal CNNs — ECG arrhythmia classifier on PhysioNet 2017
- Anchor project: multi-modal sepsis early-warning system (fuses time-series + vision)